# Fraud Detection - Final Evaluation Report

This notebook serves as the final report and presentation for the Fraud Detection project. It covers the entire pipeline from Exploratory Data Analysis (EDA) to model evaluation and final conclusions.

# 1. Executive Summary

This project aimed to develop a robust fraud detection system using the IEEE-CIS Fraud Detection dataset. The primary challenge was the extreme class imbalance and the high dimensionality of the feature space (over 400 features).

Our approach involved:
- Comprehensive data preprocessing and feature engineering.
- Dimensionality reduction using both **Mutual Information Selection** and **Principal Component Analysis (PCA)**.
- Training several models, including **XGBoost** and **Random Forest**, with specific strategies for handling class imbalance.

**Main Result:** The **XGBoost model using the Top 100 Selected Features** achieved the best balance of performance (AUC > 0.85) and efficiency, significantly reducing training time while maintaining high accuracy.

# 2. Exploratory Data Analysis (EDA) Highlights

The dataset consists of transaction and identity information. Key observations from the EDA include:
- **Class Imbalance:** Fraudulent transactions account for only ~3.5% of the data.
- **Missing Values:** Many features have high missingness, requiring careful cleaning and imputation.
- **Transaction Patterns:** Fraudulent transactions often exhibit different distribution patterns in amount and timing.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from IPython.display import Image, display
import os

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
%matplotlib inline

# Load a sample of raw data for EDA plots
# Using 100k rows for efficiency in the report
train_trans = pd.read_csv("../data/raw/train_transaction.csv", nrows=100000)

# 1. Count plot of isFraud
plt.figure(figsize=(8, 5))
sns.countplot(data=train_trans, x='isFraud', palette='viridis')
plt.title('Distribution of Fraud vs. Non-Fraud Transactions')
plt.tight_layout()
plt.show()

# 2. Bar chart of top 20 missing features
missing_values = train_trans.isnull().mean() * 100
missing_values = missing_values.sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 6))
missing_values.plot(kind='bar', color='salmon')
plt.title('Top 20 Features with Most Missing Values (%)')
plt.ylabel('Percentage Missing')
plt.tight_layout()
plt.show()

# 3. Histogram of log-transformed TransactionAmt
plt.figure(figsize=(10, 5))
sns.histplot(np.log1p(train_trans['TransactionAmt']), bins=50, kde=True, color='teal')
plt.title('Distribution of Log-Transformed Transaction Amount')
plt.xlabel('log(TransactionAmt + 1)')
plt.tight_layout()
plt.show()


# 3. Data Preprocessing

The preprocessing pipeline ensures the data is clean and suitable for machine learning:
1. **Merging:** Combined transaction and identity data on `TransactionID`.
2. **Cleaning:** Dropped features with >90% missing values.
3. **Engineering:** Created `TransactionAmt_log` and `TransactionAmt_decimal`.
4. **Imputation:** Median for numerical features, "Missing" for categorical.
5. **Encoding:** Label encoding for categorical features.
6. **Scaling:** Standardization using `StandardScaler`.
7. **Splitting:** Stratified 80/20 train-test split.


In [ ]:
# Verification: Check for missing values in processed data
X_train_processed = pd.read_parquet("../data/processed/X_train.parquet")
y_train_processed = pd.read_parquet("../data/processed/y_train.parquet")
missing_X = X_train_processed.isnull().sum().sum()
missing_y = y_train_processed.isnull().sum().sum()
print(f"X_train shape: {X_train_processed.shape}, Missing values: {missing_X}")
print(f"y_train shape: {y_train_processed.shape}, Missing values: {missing_y}")


# 4. Feature Reduction

We explored two main dimensionality reduction techniques:
- **Mutual Information (MI) Selection:** Statistical approach to select the most informative features.
- **Principal Component Analysis (PCA):** Mathematical transformation into orthogonal components.

Our final pipeline uses a consensus of **Mutual Information** and **Random Forest Importance** for the "Selected" feature set (Top 100).


In [ ]:
def display_plot(filename, title):
    path = f'../results/figures/{filename}'
    if os.path.exists(path):
        print(f"\n{title}")
        display(Image(filename=path, width=800))
    else:
        print(f"Plot {filename} not found.")

def display_cm(model_name, fs_name='selected'):
    path = f'../results/figures/cm_{model_name.lower()}_{fs_name.lower()}.png'
    if os.path.exists(path):
        print(f"\nConfusion Matrix: {model_name} ({fs_name} features)")
        display(Image(filename=path, width=500))
    else:
        print(f"Confusion matrix for {model_name} ({fs_name}) not found.")

display_plot("mi_scores.png", "Mutual Information Scores")
display_plot("combined_importance.png", "Consensus Feature Importance (MI + RF)")
display_plot("top_features_correlation.png", "Correlation Heatmap of Selected Features")
display_plot("pca_variance.png", "PCA - Cumulative Explained Variance")
display_plot("pca_loadings.png", "PCA Loadings: Original Feature Contributions")


# 5. Model Training

We compared several models across three feature sets (Full, Selected Top 100, and PCA):
- **Logistic Regression** (Baseline)
- **Decision Tree**
- **Random Forest**
- **XGBoost**
- **LightGBM**

**Imbalance Handling:**
- Used `class_weight='balanced'` for Scikit-Learn models.
- Used `scale_pos_weight=27.58` for XGBoost and LightGBM.

### Hyperparameters Summary

| Model | Key Hyperparameters |
|---|---|
| Random Forest | n_estimators=100, max_depth=10, max_features='sqrt' |
| XGBoost | n_estimators=200, max_depth=6, learning_rate=0.1, tree_method='hist' |
| LightGBM | n_estimators=100, num_leaves=31, learning_rate=0.1 |

# 6. Results and Comparison

The performance of each model configuration was evaluated using standard metrics, with a focus on AUC-ROC due to the class imbalance.


In [ ]:
metrics_path = "../results/metrics/metrics_summary.csv"
if os.path.exists(metrics_path):
    df_metrics = pd.read_csv(metrics_path)
    # Highlight best performers
    display(df_metrics.sort_values(by='AUC', ascending=False).style.background_gradient(subset=['AUC', 'F1'], cmap='YlGn'))
else:
    print("Metrics summary not found.")


In [ ]:
display_plot("model_comparison.png", "Performance Comparison Across Feature Sets")
display_plot("roc_curves.png", "ROC Curves: Impact of Feature Reduction")
display_cm("RandomForest", "selected")
display_cm("XGBoost", "selected")


# 7. Discussion

### Which reduction technique performed better?
- **Mutual Information Selection (Top 100):** This technique retained nearly all the predictive power of the full feature set while reducing dimensionality by over 75%. It is highly recommended as it preserves the original feature names, making the model more explainable.
- **PCA (95% variance):** PCA achieved similar performance levels but resulted in features that are linear combinations of the originals, which are harder to interpret for business stakeholders.

### Overall best model?
- **XGBoost** and **Random Forest** consistently achieved the highest AUC-ROC (typically > 0.85) and F1-scores. 
- The **XGBoost** model with the `hist` tree method provided the best balance of speed and accuracy.

### Business Interpretability
Using Selected Features allows the business to see which specific attributes (e.g., `TransactionAmt`, `card1`, `P_emaildomain`) are driving fraud scores, which is crucial for operationalizing the model.

# 8. Conclusion
The project successfully developed a scalable fraud detection pipeline. The final recommendation is to use the **XGBoost model with Top 100 MI-selected features**. This configuration offers excellent detection capabilities, fast processing, and maintains model explainability.

# 9. Appendix – Source Code
The full code and project history can be found at the project repository.


### How to Export this Report
To export this notebook to a professional PDF report, use the following command:
```bash
jupyter nbconvert --to pdf 05_evaluation.ipynb
```
Note: Ensure you have `nbconvert` and a TeX distribution (like MiKTeX or TeX Live) installed.
